In [1]:
from ioMicro import *

##### *Load in th parameter file for T7 transcripts*

In [22]:
import numpy as np
import pandas as pd

h_ths = np.load('T7_final_params_ths.npz')['final_params']
h_ths = h_ths.astype(int)

# create an empty dataframe with hybes (rows) and ths for each icol (columns)
master_ths = pd.DataFrame(index=range(1,8), columns=h_ths[:3,1])

# fill in the dataframe with the correct ths for each hybe/icol
for h in range(7):
    i1,i2 = h*3,(h+1)*3
    master_ths.iloc[h,:] = h_ths[int(i1):int(i2),2]

In [23]:
master_ths

,0,1,2
1,812,992,992
2,897,812,992
3,1096,1096,1212
4,1212,665,1096
5,1212,1339,1096
6,1339,1808,1339
7,1212,1212,1212


##### *Create dictionary for all T7 spots across FOVs*

In [24]:
# Use brightness thresholds loaded from master_ths (calculated separately)

T7_counts = {}

for fov in tqdm(range(225)):
    for hybe in range(1,8):
        for icol in range(3):
            # load in all fitted spots for this particular image stack
            spots = np.load(f'Z:\\Zane_20CRE\\7_17_2024__T7_20CRE\\analysis\\Conv_zscan__{fov:03d}--D{hybe:01d}_fits_icol{icol:01d}.npz')['Xh']
            
            # correct the drift for spots as they're loaded in
            drifts = np.load(f'Z:\\Zane_20CRE\\7_17_2024__T7_20CRE\\drifts_polyA\\Conv_zscan__{fov:03d}_drift.pkl', allow_pickle=True)
            drift = drifts[f'D{hybe:01d}'][0]
            spots[:,:3]-=drift
            
            # threshold spots based on brightness values from master_ths
            h = spots[:,-1]
            keep = np.where(h > master_ths.iloc[hybe-1, icol])
            spots_ = spots[keep]
            
            # update the dictionary to contain thresholded spots for this particular image stack
            T7_counts[f'{fov:03d}-D{hybe}-{icol}'] = spots_

100%|████████████████████████████████████████████████████████████████████████████████| 225/225 [04:58<00:00,  1.32s/it]


##### *Convert keys from T7_counts into the actual CRE names*

In [25]:
# make list of CRE names and filter the converted dict keys into a new list, fov_counts

CREs = ('CRE001', 'CRE002', 'CRE003', 'CRE004', 'CRE005', 'CRE006', 'CRE007', 'CRE008',
        'CRE009', 'CRE010', 'CRE011', 'CRE012', 'CRE013', 'CRE014', 'CRE015', 'CRE016',
        'CRE017', 'CRE018', 'CRE019', 'CRE020')

fov_filter = [f'{fov:03d}-D1-0', f'{fov:03d}-D1-1', f'{fov:03d}-D1-2', f'{fov:03d}-D2-0', f'{fov:03d}-D2-1', f'{fov:03d}-D2-2', f'{fov:03d}-D3-0', f'{fov:03d}-D3-1', f'{fov:03d}-D3-2', f'{fov:03d}-D4-0',
              f'{fov:03d}-D4-1', f'{fov:03d}-D4-2', f'{fov:03d}-D5-0', f'{fov:03d}-D5-1', f'{fov:03d}-D5-2', f'{fov:03d}-D6-0', f'{fov:03d}-D6-1', f'{fov:03d}-D6-2', f'{fov:03d}-D7-0', f'{fov:03d}-D7-1']

filterByKey = lambda keys: {ccre_conversion[x[3:]] : T7_counts[x] for x in keys}

ccre_conversion = {channel_id[3:] : ccre for channel_id, ccre in zip(fov_filter, CREs)}

fov_counts = []

for fov in tqdm(range(225)):
    fov_counts.append(filterByKey(fov_filter))

100%|████████████████████████████████████████████████████████████████████████████| 225/225 [00:00<00:00, 225231.12it/s]


##### *Create a cell by gene matrix for all T7 spots using the appropriate segmentation masks*

In [26]:
# Assign spots for each T7 transcript to masks generated from Cellpose and combine into a pandas dataframe

import pandas as pd
import numpy as np

masked_spots = []
all_cbg = []
fov_means = []

for fov in tqdm(range(225)):
    # load cellpose masks and add (FOV x 1000) to the mask values to distinguish them from other FOVs
    seg_mask = np.load(f'C:\\Users\\zgibbs\\cellpose\\20CRE_T7_July_2024\\20CRE_T7_July_2024_seg\\polyA\\Conv_zscan__{fov:03d}_seg.npy', allow_pickle=True)[()]
    mask_id = fov * 1000 
    seg_mask['masks'] = seg_mask['masks'] + mask_id 

    # specify the appropriate FOV for spots
    spots = fov_counts[fov]
    
    # filter spots to remove those outside the image dimensions following drift correction
    filtered_spots = {key: coords[(coords[:, 2] <= 2999) & (coords[:, 2] > 0) & (coords[:, 1] <= 2999) & (coords[:, 1] > 0), :] for key, coords in spots.items()}
    
    # round the x,y coordinates for decoded spots and assign to the masks from cellpose
    spot_masks = []
    for key in filtered_spots.keys():
        filtered_spots[key][:,2] = np.around(filtered_spots[key][:,2])
        filtered_spots[key][:,1] = np.around(filtered_spots[key][:,1])
        masks = seg_mask['masks'][np.around(filtered_spots[key][:,1]).astype(int), np.around(filtered_spots[key][:,2]).astype(int)]
        spot_masks += [np.array([masks, np.array([key] * (filtered_spots[key].shape[0])), np.around(filtered_spots[key][:,2]), np.around(filtered_spots[key][:,1])])]
        
    spot_masks = np.hstack(spot_masks).T
    barcodes = pd.DataFrame(data=spot_masks, columns=['masks', 'cre_id', 'x_round', 'y_round'])
    cell_by_gene = pd.crosstab(barcodes.masks, barcodes.cre_id)
    fov_means.append(cell_by_gene.mean(axis=0))
    all_cbg.append(cell_by_gene)
    
    # can save the cell-by-gene matrix for each FOV separately:
    # cell_by_gene.to_csv(f'/projects/ps-renlab2/zgibbs/STARR-FISH/8_23_2023__CRE-20_10XCREConc/cell_by_gene/cell_by_gene_{fov:03d}.csv')
    
all_cbg = pd.concat(all_cbg, axis=0)

100%|████████████████████████████████████████████████████████████████████████████████| 225/225 [00:37<00:00,  5.93it/s]


##### *Remove non-cell masks from the final cbg matrix*

In [27]:
non_cells = []

for fov in range(225):
    mask_id = fov * 1000
    non_cells.append(str(mask_id))
    
all_cbg.drop(labels=non_cells, axis=0, inplace = True)

##### *Add new columns to indicate total transcripts & the fov identity for each cell*

In [28]:
all_cbg['total transcripts'] = all_cbg.sum(axis=1)
all_cbg['fov'] = pd.cut(all_cbg.index.astype(int), bins = np.arange(0, 226)*1000, right=False)

In [29]:
all_cbg

cre_id,CRE001,CRE002,CRE003,CRE004,CRE005,CRE006,CRE007,CRE008,CRE009,CRE010,...,CRE013,CRE014,CRE015,CRE016,CRE017,CRE018,CRE019,CRE020,total transcripts,fov
masks,,,,,,,,,,,,,,,,,,,,,
1,0,1,0,0,0,0,0,0,0,0,...,1,0,0,0,1,0,0,0,3,"[0, 1000)"
10,2,14,16,8,7,3,12,15,1,3,...,23,1,1,1,15,13,4,7,151,"[0, 1000)"
11,1,2,2,2,0,1,1,2,0,1,...,1,0,1,1,2,0,1,2,21,"[0, 1000)"
12,6,14,21,11,6,4,10,9,3,5,...,17,2,4,2,14,8,3,6,150,"[0, 1000)"
13,7,35,33,16,11,3,12,22,2,6,...,42,2,7,5,22,21,5,11,273,"[0, 1000)"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
224058,2,4,3,2,3,1,3,3,1,2,...,2,1,1,3,1,2,2,3,41,"[224000, 225000)"
224059,0,0,1,0,0,0,0,1,0,0,...,0,0,0,0,1,0,0,0,3,"[224000, 225000)"
224060,7,10,9,10,14,3,10,9,3,11,...,13,4,5,10,11,7,11,11,171,"[224000, 225000)"


##### *Save the cell-by-gene matrix*

In [30]:
# all_cbg.to_csv(r'C:\Users\zgibbs\cellpose\cell_by_gene\SFv4_T7_July_T7_cbg.csv')